# AStrategyBaseJ

> Base class containing the core methods of CRLD agents in strategy space (JAX variant).

In [ ]:
#| default_exp Agents/StrategyBaseJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import numpy as np
import itertools as it
from functools import partial

import jax
from jax import jit
import jax.numpy as jnp
from typing import Iterable
from fastcore.utils import *

from pyCRLD.Agents.BaseJ import abaseJ
from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class strategybaseJ(abaseJ):
    """
    Base class for deterministic strategy-average independent (multi-agent)
    temporal-difference reinforcement learning in strategy space.
    """

    def __init__(self,
                 env,  # An environment object
                 learning_rates: Union[float, Iterable],  # agents' learning rates
                 discount_factors: Union[float, Iterable],  # agents' discount factors
                 choice_intensities: Union[float, Iterable] = 1.0,  # choice intensities
                 use_prefactor=False,
                 opteinsum=True,
                 **kwargs):

        self.env = env
        Tt = env.T
        # Assertion removed: jnp.allclose returns a Tracer inside JAX/numpyro
        # tracing context, which cannot be converted to a Python bool.
        Rt = env.R
        super().__init__(Tt, Rt, discount_factors, use_prefactor, opteinsum)
        self.F = jnp.array(env.F)

        self.alpha = make_variable_vectorJ(learning_rates, self.N)
        self.beta = make_variable_vectorJ(choice_intensities, self.N)
        self.TDerror = self.RPEisa

    @partial(jit, static_argnums=0)
    def step(self, Xisa) -> tuple:
        """
        Performs a learning step along the reward-prediction/temporal-difference error
        in strategy space, given joint strategy `Xisa`.
        """
        TDe = self.TDerror(Xisa)
        n = jnp.newaxis
        XexpaTDe = Xisa * jnp.exp(self.alpha[:, n, n] * TDe)
        return XexpaTDe / XexpaTDe.sum(-1, keepdims=True), TDe

    @partial(jit, static_argnums=0)
    def reverse_step(self, Xisa) -> tuple:
        """
        Performs a reverse learning step in strategy space,
        given joint strategy `Xisa`.

        This is useful to compute the separatrix of a multistable regime.
        """
        TDe = self.TDerror(Xisa)
        n = jnp.newaxis
        XexpaTDe = Xisa * jnp.exp(self.alpha[:, n, n] * -TDe)
        return XexpaTDe / XexpaTDe.sum(-1, keepdims=True), TDe

In [ ]:
#| export
@patch
def zero_intelligence_strategy(self: strategybaseJ):
    """Returns strategy `Xisa` with equal action probabilities."""
    return jnp.ones((self.N, self.Z, self.M)) / float(self.M)

In [ ]:
#| export
@patch
def random_softmax_strategy(self: strategybaseJ,
                             key: jnp.ndarray = None  # JAX PRNG key
                             ) -> jnp.ndarray:
    """Returns softmax strategy `Xisa` with random action probabilities."""
    if key is not None:
        logits = jax.random.normal(key, shape=(self.N, self.Z, self.M))
    else:
        logits = jnp.array(np.random.randn(self.N, self.Z, self.M))
    expQ = jnp.exp(logits)
    return expQ / expQ.sum(axis=-1, keepdims=True)

In [ ]:
#| export
@patch
def id(self: strategybaseJ) -> str:
    """Returns an identifier to handle simulation runs."""
    envid = self.env.id() + "__"
    agentsid = f"j{self.__class__.__name__}_"
    if hasattr(self, 'O') and hasattr(self, 'Q'):
        agentsid += 'PartObs_'
    agentsid += (f"{str(self.alpha)}_{str(self.gamma)}_{str(self.beta)}"
                 f"pre{self.use_prefactor}")
    return envid + agentsid

## Tests

In [ ]:
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ

env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)

# Create a concrete subclass to test strategybaseJ
# (RPEisa must be defined by a subclass)
class _TestStrat(strategybaseJ):
    @partial(jit, static_argnums=0)
    def RPEisa(self, Xisa, norm=False):
        return jnp.zeros_like(Xisa)  # dummy

agent = _TestStrat(env, learning_rates=0.1, discount_factors=0.9)
X0 = agent.zero_intelligence_strategy()
assert X0.shape == (2, 1, 2)
assert jnp.allclose(X0.sum(-1), 1.0)

In [ ]:
# Test random_softmax_strategy with JAX key
key = jax.random.PRNGKey(42)
Xr = agent.random_softmax_strategy(key=key)
assert Xr.shape == (2, 1, 2)
assert jnp.allclose(Xr.sum(-1), 1.0, atol=1e-5)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()